Membuat SparkSession

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

# Konfigurasi SparkSession disamakan dengan contoh gambar
spark = SparkSession.builder \
    .appName("Tugas5_BagusManggalaYudha") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


Menyiapkan Dataset

In [6]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


Instruksi

In [13]:
# 1. Baca transaksi_tugas5.csv dari HDFS
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# 2. Buat df_target dari dictionary
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

**A. Join dan Perbandingan Target** 
Ringkas total pendapatan per kota dari df_transaksi, lalu join dengan df_target. Tambahkan kolom pencapaian_persen. Urutkan hasil dari pencapaian tertinggi.

In [16]:
df_pendapatan_kota = df_transaksi.groupBy("kota").agg(spark_sum("pendapatan").alias("total_pendapatan"))
df_a = df_pendapatan_kota.join(df_target, on="kota", how="inner")
df_a = df_a.withColumn("pencapaian_persen", (col("total_pendapatan") / col("target_bulanan")) * 100)
df_a_result = df_a.orderBy(col("pencapaian_persen").desc())

df_a_result.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



**B. Window Function — Kategori Terlaris per Kota**
Menggunakan window function, tentukan kategori dengan pendapatan tertinggi di setiap kota (top-1 saja, gunakan row_number()).

In [17]:
# Hitung total pendapatan tiap kategori per kota
df_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(spark_sum("pendapatan").alias("pendapatan_kategori"))

# Tentukan window spec: partisi berdasarkan kota, urutkan berdasarkan pendapatan menurun
window_spec = Window.partitionBy("kota").orderBy(col("pendapatan**C. Spark SQL** *(bobot 25%)*

Daftarkan `df_transaksi` dan `df_target` sebagai *temporary view*, lalu **tulis satu kueri SQL** (bukan DataFrame API) yang menampilkan: `kota`, `pic_cabang`, dan jumlah transaksi (`COUNT`) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.
_kategori").desc())

# Berikan ranking dengan row_number dan ambil yang ranking 1 saja
df_ranked = df_kategori_kota.withColumn("rank", row_number().over(window_spec))
df_b_result = df_ranked.filter(col("rank") == 1).drop("rank")

df_b_result.show()

+----------+--------------------+-------------------+
|      kota|            kategori|pendapatan_kategori|
+----------+--------------------+-------------------+
|  Magelang|Kesehatan & Kecan...|            7275000|
| Purworejo|Kesehatan & Kecan...|           10075000|
|  Semarang|        Rumah Tangga|           11125000|
|      Solo|Kesehatan & Kecan...|            8425000|
|Yogyakarta|             Fashion|           13325000|
+----------+--------------------+-------------------+



**C. Spark SQL**

Daftarkan `df_transaksi` dan `df_target` sebagai *temporary view*, lalu **tulis satu kueri SQL** (bukan DataFrame API) yang menampilkan: `kota`, `pic_cabang`, dan jumlah transaksi (`COUNT`) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.

In [18]:
df_transaksi.createOrReplaceTempView("transaksi_view")
df_target.createOrReplaceTempView("target_view")

df_c_result = spark.sql('''
SELECT 
    t.kota, 
    tg.pic_cabang, 
    COUNT(*) as jumlah_transaksi
FROM transaksi_view t
JOIN target_view tg ON t.kota = tg.kota
GROUP BY t.kota, tg.pic_cabang
ORDER BY jumlah_transaksi DESC
''')
df_c_result.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



**D. Kesimpulan**

Berdasarkan hasil running kueri Spark SQL dan fungsi window di atas, kita bisa melihat perbandingan performa tiap cabang. Cabang yang berkinerja paling baik adalah Purworejo dengan PIC Fitri. Cabang ini mencatat aktivitas operasional tertinggi dengan total 116 transaksi. Selain itu, cabang Yogyakarta juga menunjukkan performa yang sangat bagus di posisi kedua dengan 110 transaksi, di mana kategori Fashion berhasil menjadi penyumbang omzet terbesar hingga Rp13.325.000.   Di sisi lain, cabang yang paling butuh dievaluasi oleh pihak manajemen adalah cabang Magelang yang dipegang oleh Rani. Cabang ini punya performa paling rendah karena jumlah transaksinya cuma 86, paling sedikit dibandingkan kota-kota lainnya. Walaupun produk Kesehatan & Kecantikan jadi yang paling laku di sana, total pendapatannya hanya menyentuh Rp7.275.000, yang mana angka ini adalah perolehan paling kecil kalau dibandingkan dengan kategori terlaris di cabang lain.   Sebagai saran, manajemen perlu memberi perhatian ekstra untuk operasional di Magelang. Karena kategori Kesehatan & Kecantikan sebenarnya sangat potensial (terbukti laku keras juga di Purworejo dan Solo), tim marketing mungkin bisa coba bikin program promo, diskon, atau bundling khusus di cabang Magelang biar traffic dan target penjualannya bisa kekejar lagi bulan depan. 